# PHYS 449 - Homework 8, Question 3

**Student Name:** Allen Mathew \
**Student ID:** 20964892 \
**Due Date:** Thursday, November 27, 3:00pm 

*ChatGPT was used to help generate the code*
- - -


3. Train three Restricted Boltzmann Machines (RBMs) on the spin configurations generated in Question 2, at $T_1 = 1.00$, $T_2 = 2.27$ (the critical temperature), and $T_3 = 3.50$. Start with 4 hidden units for the RBM, a learning rate of 10−2 or less, and train with contrastive divergence CD-k. Try $k < 10$, with a minibatch size of 100 or less, and aim for say 103 epochs of training. After you are satisfied with the training, generate new spin configurations randomly from each trained RBM. Use these new spin configurations and compute $\langle E \rangle$ and $\langle C \rangle$. Compare the results with those obtained in Question 2, and comment on the source of any discrepancies.

In [4]:
import numpy as np
import torch
import pandas as pd
import MonteCarlo as mc  # uses your HW1 Ising code: init_spins, total_energy, energy_difference, metropolis_accept, run_mcmc
from tqdm import tqdm

# =========================
# 0) Reproducibility
# =========================
SEED = 123
np.random.seed(SEED)
torch.manual_seed(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

# =========================
# 1) Generate MC configs at T1,T2,T3 (Q2-style data)
#    We add explicit warmup/equilibration (not in your run_mcmc).
# =========================
def run_mcmc_with_warmup(L, n_samples, T, J=1.0, warmup_sweeps=2000, thin=1, seed=0):
    rng = np.random.default_rng(seed)
    np.random.seed(rng.integers(0, 2**31-1))  # since mc.* uses global np.random

    spins = mc.init_spins(L)
    N = L * L
    E = mc.total_energy(spins, L, J)

    # warmup
    for _ in range(warmup_sweeps):
        for _ in range(N):
            x, y = np.random.randint(0, L, size=2)
            dE = mc.energy_difference(spins, L, J, x, y)
            if mc.metropolis_accept(dE, T):
                spins[x, y] *= -1
                E += dE

    # sampling
    samples = np.empty((n_samples, L, L), dtype=np.int8)
    energies = np.empty(n_samples, dtype=np.float64)

    for i in range(n_samples):
        for _ in range(thin):
            for _ in range(N):
                x, y = np.random.randint(0, L, size=2)
                dE = mc.energy_difference(spins, L, J, x, y)
                if mc.metropolis_accept(dE, T):
                    spins[x, y] *= -1
                    E += dE
        samples[i] = spins
        energies[i] = E

    return samples, energies

def ising_stats_from_energies(E, T, Nspins):
    Emean = E.mean()
    E2mean = (E**2).mean()
    C = (E2mean - Emean**2) / (T**2)          # total C
    return (Emean / Nspins, C / Nspins)       # per spin

# =========================
# 2) RBM (binary visible & hidden) + CD-k training
#    spins {-1,+1} mapped to bits {0,1}
# =========================
def spins_to_bits(spins_pm1):
    return ((spins_pm1 + 1) // 2).astype(np.float32)

def bits_to_spins(bits01):
    return (2*bits01 - 1).astype(np.int8)

class RBM(torch.nn.Module):
    def __init__(self, n_visible, n_hidden, seed=0):
        super().__init__()
        g = torch.Generator(device="cpu").manual_seed(seed)
        self.W = torch.nn.Parameter(0.01 * torch.randn(n_visible, n_hidden, generator=g))
        self.a = torch.nn.Parameter(torch.zeros(n_visible))  # visible bias
        self.b = torch.nn.Parameter(torch.zeros(n_hidden))   # hidden bias

    def p_h_given_v(self, v):
        return torch.sigmoid(v @ self.W + self.b)

    def p_v_given_h(self, h):
        return torch.sigmoid(h @ self.W.t() + self.a)

    @torch.no_grad()
    def sample_bernoulli(self, p):
        return (torch.rand_like(p) < p).float()

    @torch.no_grad()
    def gibbs_step(self, v):
        ph = self.p_h_given_v(v)
        h  = self.sample_bernoulli(ph)
        pv = self.p_v_given_h(h)
        v2 = self.sample_bernoulli(pv)
        return v2

    @torch.no_grad()
    def gibbs_k(self, v, k=1):
        for _ in range(k):
            v = self.gibbs_step(v)
        return v

def train_rbm_cd_k(rbm, X_bits, lr=1e-2, batch_size=100, k=5, epochs=1000, seed=0, print_every=100):
    """
    CD-k with the standard approximate gradient:
      dW ~ v0^T ph0 - vk^T phk
      da ~ <v0 - vk>, db ~ <ph0 - phk>
    plus a simple training diagnostic: mean reconstruction error ||v0 - vk||_1.
    """
    rng = np.random.default_rng(seed)
    X = torch.from_numpy(X_bits).to(device)
    M = X.shape[0]
    rbm.to(device)

    for ep in range(1, epochs+1):
        perm = rng.permutation(M)
        Xp = X[perm]

        recon_err_epoch = 0.0
        nb = 0

        for start in range(0, M, batch_size):
            v0 = Xp[start:start+batch_size]
            if v0.shape[0] == 0:
                continue

            ph0 = rbm.p_h_given_v(v0)
            vk  = rbm.gibbs_k(v0, k=k)
            phk = rbm.p_h_given_v(vk)

            dW = (v0.t() @ ph0 - vk.t() @ phk) / v0.shape[0]
            da = (v0 - vk).mean(dim=0)
            db = (ph0 - phk).mean(dim=0)

            rbm.W.data += lr * dW
            rbm.a.data += lr * da
            rbm.b.data += lr * db

            recon_err_epoch += (v0 - vk).abs().mean().item()
            nb += 1

        if ep % print_every == 0:
            print(f"epoch {ep:4d}/{epochs} | mean |v0-vk| = {recon_err_epoch/max(nb,1):.4f}")

    return rbm

# =========================
# 3) "Generate new spin configurations randomly from each trained RBM"
#    = initialize v ~ Bernoulli(0.5) (random), then Gibbs burn-in, then sample with thinning.
# =========================
@torch.no_grad()
def rbm_generate_samples(rbm, n_samples, burn_in=2000, between=10, seed=0):
    torch.manual_seed(seed)
    nV = rbm.a.numel()
    v = (torch.rand(1, nV, device=device) < 0.5).float()  # random start

    # burn-in
    for _ in range(burn_in):
        v = rbm.gibbs_step(v)

    out = torch.empty(n_samples, nV, device="cpu")
    for i in range(n_samples):
        for _ in range(between):
            v = rbm.gibbs_step(v)
        out[i] = v.detach().cpu()[0]
    return out.numpy()

# =========================
# 4) Run Q3: Train at T1,T2,T3; generate; compute <E>, <C>; compare to MC
# =========================
L = 16
J = 1.0
Nspins = L*L

T_list = [1.00, 2.27, 3.50]

# keep MC dataset moderate so 10^3 epochs is practical
n_train = 10000
warmup_sweeps = 2000
thin_train = 1

# RBM hyperparams required by prompt
n_hidden = 4
lr = 1e-2              # "10^-2 or less"
k_cd = 5               # "Try k < 10"
batch_size = 100       # "100 or less"
epochs = 1000          # "aim for say 10^3 epochs"

# RBM generation settings
n_gen = 10000
burn_in = 2000
between = 10

results = []

for idx, T in enumerate(T_list):
    print("\n" + "="*80)
    print(f"Training RBM for T={T:.2f}")

    # --- MC data (this is what you trained on in Q2, but only at these 3 temperatures) ---
    X_spins, E_series = run_mcmc_with_warmup(
        L=L, n_samples=n_train, T=T, J=J,
        warmup_sweeps=warmup_sweeps, thin=thin_train, seed=10+idx
    )
    E_mc, c_mc = ising_stats_from_energies(E_series, T, Nspins)
    print(f"MC baseline:  <E>/N={E_mc:.6f},  c={c_mc:.6f}")

    # --- RBM training ---
    X_bits = spins_to_bits(X_spins).reshape(n_train, -1)
    rbm = RBM(n_visible=L*L, n_hidden=n_hidden, seed=100+idx)
    rbm = train_rbm_cd_k(
        rbm, X_bits,
        lr=lr, batch_size=batch_size, k=k_cd, epochs=epochs,
        seed=200+idx, print_every=100
    )

    # --- Generate NEW configurations from RBM (random start + Gibbs) ---
    gen_bits = rbm_generate_samples(rbm, n_samples=n_gen, burn_in=burn_in, between=between, seed=300+idx)
    gen_spins = bits_to_spins(gen_bits.reshape(n_gen, L, L))

    # --- Compute <E>, <C> from generated configs using the SAME Ising energy function ---
    # Use your mc.total_energy for consistency (loop is OK for 10k)
    def energy_batch_safe(spins_pm1, J=1.0):
        s = np.asarray(spins_pm1).astype(np.int32, copy=False)

        if s.ndim == 2:
            # single config
            E = (s * np.roll(s, -1, axis=0)).sum(dtype=np.int64)
            E += (s * np.roll(s, -1, axis=1)).sum(dtype=np.int64)
            return float(-J * E)

        # batch
        E = (s * np.roll(s, -1, axis=1)).sum(axis=(1,2), dtype=np.int64)
        E += (s * np.roll(s, -1, axis=2)).sum(axis=(1,2), dtype=np.int64)
        return (-J * E).astype(np.float64)


    E_gen = energy_batch_safe(gen_spins, J=J)
    E_rbm, c_rbm = ising_stats_from_energies(E_gen, T, Nspins)
    print(f"RBM samples:  <E>/N={E_rbm:.6f},  c={c_rbm:.6f}")

    results.append({
        "T": T,
        "MC <E>/N": E_mc, "MC c": c_mc,
        "RBM <E>/N": E_rbm, "RBM c": c_rbm,
        "Δ<E>/N": E_rbm - E_mc,
        "Δc": c_rbm - c_mc
    })

df = pd.DataFrame(results)
print("\n=== Comparison table (RBM vs MC baseline) ===")
display(df)


device: cpu

Training RBM for T=1.00
MC baseline:  <E>/N=-1.997453,  c=0.021227
epoch  100/1000 | mean |v0-vk| = 0.0026
epoch  200/1000 | mean |v0-vk| = 0.0016
epoch  300/1000 | mean |v0-vk| = 0.0012
epoch  400/1000 | mean |v0-vk| = 0.0010
epoch  500/1000 | mean |v0-vk| = 0.0009
epoch  600/1000 | mean |v0-vk| = 0.0009
epoch  700/1000 | mean |v0-vk| = 0.0008
epoch  800/1000 | mean |v0-vk| = 0.0008
epoch  900/1000 | mean |v0-vk| = 0.0008
epoch 1000/1000 | mean |v0-vk| = 0.0007
RBM samples:  <E>/N=-1.996545,  c=0.027926

Training RBM for T=2.27
MC baseline:  <E>/N=-1.467912,  c=1.481568
epoch  100/1000 | mean |v0-vk| = 0.3731
epoch  200/1000 | mean |v0-vk| = 0.3819
epoch  300/1000 | mean |v0-vk| = 0.3144
epoch  400/1000 | mean |v0-vk| = 0.2888
epoch  500/1000 | mean |v0-vk| = 0.2750
epoch  600/1000 | mean |v0-vk| = 0.2612
epoch  700/1000 | mean |v0-vk| = 0.2542
epoch  800/1000 | mean |v0-vk| = 0.2615
epoch  900/1000 | mean |v0-vk| = 0.2589
epoch 1000/1000 | mean |v0-vk| = 0.2516
RBM sampl

,T,MC <E>/N,MC c,RBM <E>/N,RBM c,Δ<E>/N,Δc
0,1.00,-1.997453,0.021227,-1.996545,0.027926,0.000908,0.006699
1,2.27,-1.467912,1.481568,-0.737769,0.894412,0.730144,-0.587156
2,3.50,-0.661433,0.237531,-0.091166,0.195549,0.570267,-0.041982


After training the 3 RBMs on Monte Carlo spinn  at $T = 1.00, 2.27, and 3.50$, we can see that there are some differences in their respective $\langle E \rangle$ and $\langle C \rangle$ values. At $T=1.00$, the RBM actaully reproduces the Monte Carlo values very well. The decrepencies between their respective expectation values are in the order of $10^{-3}$ to $10^{-4}$. This makese sense in low temperatures, the distribution is dominated by strongly ordered configurations. At $T=2.27$, we get the highest discrepancies in the order of $10^{-1}$. This is expected since this is the critical temperature, thus the data becomes more broad. The RBM only has 4 hidden units, so it becomes difficult to capture the variability of the data at critical temperature. At $T=3.50$, the discrepencies lower to about an order of $10^{-1}$ to $10^{-2}$. This could be caused by the fact that the RBM is undermodelling the remaining nearest-neighbor correlations present at $T=3.50$. Overall, the discrepencies are mainly due to the very small hidden layer and the fact that matching critical/near-critical correlations is substantially harder than matching strongly ordered low $T$ data.